In [33]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom

--------
## Setup: Create the modules for the BAPM Simulation
--------
Here

In [34]:
class BAPM:
    '''
    This class contains the functions needed to implement the BAPM.
    '''
    
    def __init__(self, S0, r, u, d, N):
        '''
        Initialize the BAPM model with the following parameters:
        -----------------------------
        S0: Initial stock price
        r: Risk-free rate
        u: Up factor
        d: Down factor
        N: Number of time steps
        '''
        self.S0 = S0
        self.r = r
        self.u = u
        self.d = d
        self.N = N
        # Calculate the risk-neutral probability
        self.p_tild = self.risk_neutral_prob()
        # Quick no-arbitrage check
        if not self.check_no_arbitrage():
            print("WARNING: Parameters don't satisfy no-arbitrage constraint")
            print(f"Require d < 1+r < u but have: {d} < {1+r} < {u}")

    # %==========================================%
    # Begin definition of helper functions
    # %==========================================%
    def check_no_arbitrage(self):
        # Check no-arbitrage constraint from bapm_01 pg. 20
        return self.d < 1 + self.r < self.u
    def risk_neutral_prob(self):
        return ((1 + self.r) - self.d) / (self.u - self.d)

    # Calculate the possible stock prices at the terminal time N
    def compute_terminal_stock_price(self):
        # For a path-independent derivative, we only need the distribution of SN
        # There are N+1 possible values for SN
        S = np.zeros(self.N+1)
        for i in range(self.N+1):
            # i is the number of up moves in N steps
            S[i] = self.S0 * (self.u ** i) * (self.d **(self.N-i))
            
        return S

    def compute_expectation(self, payoffs, p):
        '''
        Compute E_p[Ṽ_n] for given probability p
        payoff: Payoff values corresponding to stock price outcomes
        p: Probability of 'up'-step
        '''
        expectation = 0
        discount_factor = 1 / ((1+self.r) ** self.N)
        for i in range(self.N+1):
            # Probability of exactly i up moves within N steps under the given 
            # probability measure p
            prob = binom.pmf(i, self.N, p)
            expectation += prob * payoffs[i]

        return expectation * discount_factor
    
    def replicating_portfolio_step(self, V_up, V_down, S_up, S_down):
        '''
        Compute one step of the replicating portfolio
        
        V_up: Derivative value if the stock goes up
        V_down: Derivative value if stock goes down
        S_up: Stock price if it geos up
        S_down: stock price if it goes down
        '''
        # Solve for delta from the wealth equations:
        # V_up = Delta * S_up + (1+r) * (V - Delta * S)
        # V_down = Delta * S_down + (1+r) * (V - Delta * S)
        Delta = (V_up - V_down) / (S_up - S_down)

        # Current stock price before up/down move
        S_current = S_down / self.d
        
        # Substitute:
        V = (V_up - Delta * S_up) / (1+self.r) + Delta * S_current
        # Return number of shares and value of derivative
        return Delta, V
    
    def compute_replicating_portfolio(self, payoff_func):
        '''
        payoff_func: Function that takes stock price and returns payoff
        '''
        V = {}
        Delta = {}
        # Compute all possible terminal stock prices:
        terminal_stocks = self.compute_terminal_stock_price()

        # Now compute the terminal payoffs at time step N
        V[self.N] = np.array([payoff_func(S) for S in terminal_stocks])

        # Extend one-step routine by running backwards to derive X0=V0
        for n in range(self.N-1, -1, -1):
            V[n] = np.zeros(n+1)
            Delta[n] = np.zeros(n+1)

            for j in range(n+1):
                # The current stock price after 'j' up-moves in n steps
                S_j = self.S0 * (self.u ** j) * (self.d **(n-j))
                # Stock price at n+1 for up and down cases:
                S_up = S_j * self.u
                S_down = S_j * self.d
                # Value at n+1 for up and down cases:
                V_up = V[n+1][j+1]  # j+1 up moves in n+1 steps
                V_down = V[n+1][j]  # j up moves in n+1 steps
                # Compute delta and the value at this node
                Delta[n][j], V[n][j] = self.replicating_portfolio_step(V_up, V_down, S_up, S_down)

        # Initial value V0 is the only element in V[0]:
        V0 = V[0][0]
        # Return the inital value, dictionary of deltas, and value at each node
        return V0, Delta, V
    
    # Code to analyze the portfolio positions for Q1.PartD.2
    def analyze_portfolio_positions(self, Delta, V):
        # Params:
        # V: Dictionary of derivative values at each node

        # Initialize the dictionary of results to send back
        results = {
            'short_selling': False,
            'borrowing': False,
            'nodes': []
        }
        for n in range(self.N):
            for j in range(n+1):
                # Calculate the current stock price:
                S_j = self.S0 * (self.u ** j) * (self.d ** (n-j))

                # Compute the stock and money market positions as described in 
                # the problem statement
                stock_position = Delta[n][j] * S_j
                portfolio_value = V[n][j]
                money_market = portfolio_value - stock_position

                # Check for short-selling or borrowing by analyzing weights:
                if Delta[n][j] < 0:
                    results['short_selling'] = True
                if money_market < 0:
                    results['borrowing'] = True

                # Add additional information for this node
                results['nodes'].append({
                    'time': n,
                    'up-moves': j,
                    'stock_price': S_j,
                    'delta': Delta[n][j],
                    'stock_position': stock_position,
                    'portfolio_value': portfolio_value,
                    'money_market': money_market,
                    'short_selling': Delta[n][j] < 0,
                    'borrowing': money_market < 0
                })
                
        return results
    
    def is_delta_path_independent(self, Delta):
        ''' Delta is path-independent if it depends only on the stock
            price at each node, not on specific path to reach that price
        '''
        # So we check each time step n and create a dictionary that 
        # stores the delta values which are keyed by the stock price
        for n in range(1, self.N):
            delta_by_price = {}
            for j in range(n+1):
                S = self.S0 * (self.u ** j) * (self.d ** (n - j))
                S_rounded = round(S, 10)
                # Check if price has been seen before:
                if S_rounded in delta_by_price:
                    # Check if the delta is the same
                    if not np.isclose(Delta[n][j], delta_by_price[S_rounded]):
                        return False
                else: 
                    # Add the price to the dictionary
                    delta_by_price[S_rounded] = Delta[n][j]
        return True

--------------
## Question 1 Part (d) - Test specific example
--------------
Now that the above BAPM class and associated functions are created we test them by running a specific example. It is instructed to take r = 0.05, u = 1.1, d = 1.01, N = 5, and a derivative being a European call option with strike price K = (1+r)^N S_0

In [35]:
def simulate_BAPM(S0=1.0, r=0.05, u=1.1, d=1.01, N=5):
    print("BAPM Model Simulation ")
    print("---------------------")

    # Pull the BAPM model 
    model = BAPM(S0, r, u, d, N)
    print(f"BAPM model parameters: S0={S0}, r={r}, u={u}, d={d}, N={N}")
    print(f"Risk-neutral probability p̃ = {model.p_tild:.6f}")

    # define the European call option
    K = ((1+r) ** N) * S0
    print(f"Strike price K = {K:.6f}")

    def call_payoff(S):
        return max(0, S-K)
    
    # Compute terminal stock prices and payoffs
    terminal_prices = model.compute_terminal_stock_price()
    terminal_payoffs = np.array([call_payoff(S) for S in terminal_prices])

    print("\nTerminal stock prices and payoffs:")
    for j, (price, payoff) in enumerate(zip(terminal_prices, terminal_payoffs)):
        print(f"    {j} up moves: S={price:.6f}, V={payoff:.6f}")

    # Part (a): Compute expectations under different probabilities
    p1 = min(model.p_tild + 0.1, 0.99)
    p2 = max(model.p_tild - 0.1, 0.01)
    p_tild = model.p_tild

    print(f"\n Probabilities for testing:")
    print(f"p1 = {p1:.4f} (> p̃)")
    print(f"p2 = {p2:.4f} (< p̃)")
    print(f"p̃ = {p_tild:.4f}")

    # Compute and display the expectations:
    exp_p1 = model.compute_expectation(terminal_payoffs, p1)
    exp_p2 = model.compute_expectation(terminal_payoffs, p2)
    exp_p_tild = model.compute_expectation(terminal_payoffs, p_tild)

    print("\nExpected values of discounted payoff:")
    print(f"E_p1[Ṽ_N] = {exp_p1:.4f}")
    print(f"E_p2[Ṽ_N] = {exp_p2:.4f}")
    print(f"E_p̃[Ṽ_N] = {exp_p_tild:.4f}")

    return model, call_payoff, exp_p_tild
    # ======================
    # END OF BAPM SIMULATION
    # ======================

def test_replicating_portfolio(model, payoff_func, expected_value):
    print("\nREPLICATING PORTFOLIO TESTS")
    print("-----------------------------")
    # Compute the replicating portfolio
    V0, Delta, V = model.compute_replicating_portfolio(payoff_func)
    print(f"Initial value from replicating portfolio: V0 = {V0:.4f}")
    print(f"Expected value from risk-neutral measure: {expected_value:.4f}")
    print(f"Difference: {abs(V0 - expected_value):.6f}")

    # Check if deltas are path independent part (c) 1. :
    is_path_independent = model.is_delta_path_independent(Delta)
    print(f"\nIs Delta path independent? {is_path_independent}")

    # Analyze portfolio positions for part (d) 2. :
    portfolio_analysis = model.analyze_portfolio_positions(Delta, V)

    print("\nPortfolio Position Analysis:")
    print(f"Short selling required? {portfolio_analysis['short_selling']}")
    print(f"Borrowing required? {portfolio_analysis['borrowing']}")

    print("\nDetailed portfolio positions (selected nodes):")
    for node in portfolio_analysis['nodes'][:min(6, len(portfolio_analysis['nodes']))]:
        print(f"Time {node['time']}, {node['up-moves']} up moves:")
        print(f"  Stock price: {node['stock_price']:.6f}")
        print(f"  Delta: {node['delta']:.6f}")
        print(f"  Stock position value: {node['stock_position']:.6f}")
        print(f"  Portfolio value: {node['portfolio_value']:.6f}")
        print(f"  Money market position: {node['money_market']:.6f}")
        print(f"  {'Short selling' if node['short_selling'] else 'Long position'} in stock")
        print(f"  {'Borrowing' if node['borrowing'] else 'Lending'} in money market")
  
    return V0, Delta, V, portfolio_analysis    

In [36]:
# Run the BAPM simulation:
model, payoff_func, expected_value = simulate_BAPM()
# Check the replicating portfolio 
V0, Delta, V, analysis = test_replicating_portfolio(model, payoff_func, expected_value)


BAPM Model Simulation 
---------------------
BAPM model parameters: S0=1.0, r=0.05, u=1.1, d=1.01, N=5
Risk-neutral probability p̃ = 0.444444
Strike price K = 1.276282

Terminal stock prices and payoffs:
    0 up moves: S=1.051010, V=0.000000
    1 up moves: S=1.144664, V=0.000000
    2 up moves: S=1.246664, V=0.000000
    3 up moves: S=1.357753, V=0.081472
    4 up moves: S=1.478741, V=0.202459
    5 up moves: S=1.610510, V=0.334228

 Probabilities for testing:
p1 = 0.5444 (> p̃)
p2 = 0.3444 (< p̃)
p̃ = 0.4444

Expected values of discounted payoff:
E_p1[Ṽ_N] = 0.0657
E_p2[Ṽ_N] = 0.0198
E_p̃[Ṽ_N] = 0.0390

REPLICATING PORTFOLIO TESTS
-----------------------------
Initial value from replicating portfolio: V0 = 0.0390
Expected value from risk-neutral measure: 0.0390
Difference: 0.000000

Is Delta path independent? True

Portfolio Position Analysis:
Short selling required? False
Borrowing required? True

Detailed portfolio positions (selected nodes):
Time 0, 0 up moves:
  Stock price: 1.0

---------------
## Question 1 Results
---------------
1. Which case gives the correct value for S0 and V0?
    - The risk-neutral measure of 0.4444 gives the correct value of V0. The initial value from the replicating portfolio (V0 = 0.0390) exactly matches the expected value under the risk-neutral measure.
    - The incorrect values show risk premiums since for p1 and p2 the expectation under these measures is either greater or less than the expectation under the risk-neutral measure. For example, p1 > p_tild: E_p1[V] = 0.0657 > E_p_tild. Therefore this shows a positive risk premium i.e. using a higher probability of up-moves results in a higher expected payoff. 

2. Any short selling or borrowing?
    - No short selling (all deltas are greater than zero)
    - Yes for borrowing (all money market positions are negative)
    - What this means is that at every node in the tree we should buy and hold a positive amount of the stock, and borrow money to finance this stock purchase. This is the hedging strategy to ensure the portfolio replicates the derivative's value at each step

3. Is delta path independent?
    - Yes

In [37]:
# %===============================================================%
# Monte Carlo implementation for the Binomial Asset Pricing Model
# %===============================================================%
class BAPMMonteCarlo:
    def __init__(self, S0, r, u, d, N):
        # Initialize BAPM model with associated parameters
        self.S0 = S0
        self.r = r
        self.u = u
        self.d = d
        self.N = N
        
        # Calculate risk-neutral probability
        self.p_tilde = self.calculate_risk_neutral_prob()
        
        if not self.check_no_arbitrage():
            print("WARNING: Parameters don't satisfy no-arbitrage constraint")
            print(f"Require d < 1+r < u but have: {d} < {1+r} < {u}")
    
    def check_no_arbitrage(self):
        return self.d < 1 + self.r < self.u    
    def calculate_risk_neutral_prob(self):
        return ((1 + self.r) - self.d) / (self.u - self.d)
    
    def compute_stock_price_from_path(self, path):
        """
        Compute the stock price given a path of coin tosses
        
        Parameters:
        path: List of 'H' and 'T' representing up and down moves
        """
        S = self.S0
        for move in path:
            if move == 'H':
                S *= self.u
            else:
                S *= self.d
        # Stock price after following the path:
        return S
    
    def generate_random_path(self, n, p):
        # Random list of length n of heads and tails according to prob p (up-move)
        return ['H' if np.random.random() < p else 'T' for _ in range(n)]
    
    def monte_carlo_expectation(self, n, N, M, omega_n=None, payoff_func=None, p=None):
        """
        Compute E_p[X_{n+m}|F_n] using Monte Carlo simulation
        
        n: Current time step
        N: Terminal time step
        M: Number of Monte Carlo paths
        omega_n: Path up to time n (list of 'H' and 'T')
        payoff_func: Payoff function to apply at time N
        p (float, optional): Probability of up move (defaults to risk-neutral)
        """
        if p is None:
            p = self.p_tilde  # Default to risk-neutral probability
        
        if omega_n is None:
            omega_n = []  # Empty path means starting from time 0
        
        # Compute current stock price S_n from omega_n
        S_n = self.compute_stock_price_from_path(omega_n)
        
        # Generate M paths  according to p tilde and compute
        # U(omega^m) for each path
        results = []
        for _ in range(M):
            # Start with current stock price
            S_current = S_n
            
            # Generate random extension from n to N
            for _ in range(N - n):
                if np.random.random() < p:
                    S_current *= self.u  # Up move
                else:
                    S_current *= self.d  # Down move
            
            # Apply payoff function or return terminal stock price
            if payoff_func:
                payoff = payoff_func(S_current)
                # Apply discounting from time N back to time n
                discounted_payoff = payoff / ((1 + self.r) ** (N - n))
                results.append(discounted_payoff)
            else:
                results.append(S_current)
        
        # Return average (expected value)
        return np.mean(results)
    
    def european_call_payoff(self, S, K):
        return max(0, S - K)
    
    def lookback_option_payoff(self, path):
        """
        Compute the payoff for a lookback option
        
        V_N = max_(0≤n≤N) (1+r)^(N-n) S_n - S_N
        """
        # Compute the stock price at each time step
        S = [self.S0]
        for move in path:
            if move == 'H':
                S.append(S[-1] * self.u)
            else:
                S.append(S[-1] * self.d)
        
        # Compute the discounted maximum
        max_discounted = 0
        for n in range(len(S)):
            discounted_value = (1 + self.r) ** (self.N - n) * S[n]
            max_discounted = max(max_discounted, discounted_value)
        
        # Return the payoff
        return max_discounted - S[-1]
    
    # Monte Carlo simulation for the lookback option
    def monte_carlo_lookback(self, n, N, M, omega_n=None, p=None):

        if p is None:
            p = self.p_tilde
        
        if omega_n is None:
            omega_n = []
        
        # Compute S_n and M_n (maximum discounted value up to time n)
        S_n = self.S0
        M_n = self.S0  # Initial value
        
        for i, move in enumerate(omega_n):
            if move == 'H':
                S_n *= self.u
            else:
                S_n *= self.d
            
            # Update maximum discounted value
            discounted_value = (1 + self.r) ** (n - i - 1) * S_n
            M_n = max(M_n, discounted_value)
        
        # Generate M random path extensions
        results = []
        for _ in range(M):
            # Start with current stock price and max value
            S_current = S_n
            M_current = M_n
            
            # Generate random extension from n to N
            for i in range(N - n):
                if np.random.random() < p:
                    S_current *= self.u
                else:
                    S_current *= self.d
                
                # Update maximum discounted value
                time_step = n + i + 1
                discounted_value = (1 + self.r) ** (N - time_step) * S_current
                M_current = max(M_current, discounted_value)
            
            # Compute payoff: max discounted - final price
            payoff = M_current - S_current
            
            # Apply discounting from time N back to time n
            discounted_payoff = payoff / ((1 + self.r) ** (N - n))
            results.append(discounted_payoff)
        
        # Return average
        return np.mean(results)


    def test_monte_carlo_convergence(self, M_values, n=0, omega_n=None, option_type="stock", repeat=1):
        """
        Test Monte Carlo convergence for different M values:
        
        M_values: List of different M values to test
        n: Starting time step
        option_type: stock/call/lookback
        repeat: Number of times to repeat each test for variance estimation
        """
        results = {
            "M_values": M_values,
            "means": [],
            "variances": []
        }
        
        for M in M_values:
            # Repeat the Monte Carlo estimation multiple times
            estimates = []
            
            for _ in range(repeat):
                if option_type == "stock":
                    # Estimate S_N
                    estimate = self.monte_carlo_expectation(n, self.N, M, omega_n)
                elif option_type == "call":
                    # Estimate V_0 for European call
                    K = (1 + self.r) ** self.N * self.S0
                    payoff_func = lambda S: self.european_call_payoff(S, K)
                    estimate = self.monte_carlo_expectation(n, self.N, M, omega_n, payoff_func)
                elif option_type == "lookback":
                    # Estimate V_0 for lookback option
                    estimate = self.monte_carlo_lookback(n, self.N, M, omega_n)
                
                estimates.append(estimate)
            
            # Compute mean and variance of the estimates
            results["means"].append(np.mean(estimates))
            results["variances"].append(np.var(estimates))
        
        return results

In [38]:
# Part 2(a): Test Monte Carlo with N=5
def test_monte_carlo_N5():
    """Test Monte Carlo with N=5 and different M values"""
    print("\nPART 2(a): MONTE CARLO WITH N=5")
    print("-------------------------------")
    
    # Set parameters
    S0 = 1.0
    r = 0.05
    u = 1.1
    d = 1.01
    N = 5
    
    # Initialize model
    model = BAPMMonteCarlo(S0, r, u, d, N)
    print(f"BAPM for Monte Carlo, Parameters: S0={S0}, r={r}, u={u}, d={d}, N={N}")
    
    # Define M values to test
    M_values = [1, 5, 10, 32, 100, 1000]
    
    # Test S0 estimation:
    print("\nEstimating S0 from Monte Carlo:")
    stock_results = model.test_monte_carlo_convergence(M_values, option_type="stock", repeat=10)
    
    print(f"{'M':>10} {'S0 Estimate':>15} {'Error':>10} {'Variance':>10}")
    for i, M in enumerate(M_values):
        estimate = stock_results["means"][i]
        variance = stock_results["variances"][i]
        error = abs(estimate - S0)
        print(f"{M:10d} {estimate:15.6f} {error:10.6f} {variance:10.6f}")
    
    # European call option
    K = (1+r)**N * S0
    
    def call_payoff(S):
        return max(0, S - K)
    
    # Compute exact price
    terminal_prices = np.zeros(N+1)
    for i in range(N+1):
        terminal_prices[i] = S0 * (u ** i) * (d ** (N-i))
    
    terminal_payoffs = np.array([call_payoff(S) for S in terminal_prices])
    
    # Comopute exact price with the risk neutral measure
    expectation = 0
    p_tilde = model.p_tilde
    discount_factor = 1 / ((1 + r) ** N)
    
    for i in range(N+1):
        prob = binom.pmf(i, N, p_tilde)
        expectation += prob * terminal_payoffs[i]
    
    exact_V0 = expectation * discount_factor
    print(f"\nExact option price V0 = {exact_V0:.6f}")
    
    # Test V0 estimation
    print("\nEstimating V0 from Monte Carlo:")
    option_results = model.test_monte_carlo_convergence(M_values, option_type="call", repeat=10)
    
    # Print results
    print(f"{'M':>10} {'V0 Estimate':>15} {'Error':>10} {'Variance':>10}")
    for i, M in enumerate(M_values):
        estimate = option_results["means"][i]
        variance = option_results["variances"][i]
        error = abs(estimate - exact_V0)
        print(f"{M:10d} {estimate:15.6f} {error:10.6f} {variance:10.6f}")


# Part 2(b): Monte Carlo with N=100
def test_monte_carlo_N100():
    """Test Monte Carlo with N=100 and different M values"""
    print("\nPART 2(b): MONTE CARLO WITH N=100, r=10^-3, u = 1+5x10^-3, d=1+10^-4")
    print("--------------------------------")
    
    # Set parameters
    S0 = 1.0
    r = 0.001
    u = 1.005
    d = 1.0001
    N = 100
    
    # Initialize model
    model = BAPMMonteCarlo(S0, r, u, d, N)
    print(f"Model parameters: S0={S0}, r={r}, u={u}, d={d}, N={N}")
    print(f"Risk-neutral probability p̃ = {model.p_tilde:.6f}")
    
    # Define M values to test
    M_values = [10, 100, 1000, 10000]
    
    # Test S0 estimation
    print("\nEstimating S0 from Monte Carlo:")
    stock_results = model.test_monte_carlo_convergence(M_values, option_type="stock", repeat=5)
    
    # Print results
    print(f"{'M':>10} {'S0 Estimate':>15} {'Error':>10} {'Variance':>10}")
    for i, M in enumerate(M_values):
        estimate = stock_results["means"][i]
        variance = stock_results["variances"][i]
        error = abs(estimate - S0)
        print(f"{M:10d} {estimate:15.6f} {error:10.6f} {variance:10.6f}")
    
    K = (1+r)**N * S0
    print(f"Strike price K = {K:.6f}")
    
    print("\nEstimating V0 from Monte Carlo:")
    option_results = model.test_monte_carlo_convergence(M_values, option_type="call", repeat=5)
    
    print(f"{'M':>10} {'V0 Estimate':>15} {'Variance':>10}")
    for i, M in enumerate(M_values):
        estimate = option_results["means"][i]
        variance = option_results["variances"][i]
        print(f"{M:10d} {estimate:15.6f} {variance:10.6f}")

# Part 2(c): Generate random paths and test Monte Carlo
def test_monte_carlo_random_paths():
    """
    Generate 5 random paths of length 10 and test Monte Carlo for each
    """
    print("\nPART 2(c): MONTE CARLO WITH RANDOM PATHS")
    print("---------------------------------------")
    
    # Set parameters
    S0 = 1.0
    r = 0.001
    u = 1.005
    d = 1.0001
    N = 100
    n = 10  # Length of initial paths
    
    # Initialize model
    model = BAPMMonteCarlo(S0, r, u, d, N)
    
    # Generate 5 random paths with p=0.9
    p_actual = 0.9
    random_paths = [model.generate_random_path(n, p_actual) for _ in range(5)]
    
    # For each path, compute S_10 and use Monte Carlo to estimate S_10 and V_10
    print("\nResults for 5 random paths:")
    print(f"{'Path':>5} {'Exact S_10':>15} {'M':>10} {'MC S_10':>15} {'Error':>10} {'MC V_10':>15}")
    
    M_values = [10, 100, 1000]
    
    for i, path in enumerate(random_paths):
        # Compute exact S_10
        exact_S_10 = model.compute_stock_price_from_path(path)
        
        for M in M_values:
            # Estimate S_10 using Monte Carlo
            S_10_estimate = model.monte_carlo_expectation(n, N, M, path)
            
            # Estimate V_10 using Monte Carlo
            K = (1+r)**N * S0
            payoff_func = lambda S: model.european_call_payoff(S, K)
            V_10_estimate = model.monte_carlo_expectation(n, N, M, path, payoff_func)
            
            # Compute error
            error = abs(S_10_estimate - exact_S_10)
            
            print(f"{i+1:5d} {exact_S_10:15.6f} {M:10d} {S_10_estimate:15.6f} {error:10.6f} {V_10_estimate:15.6f}")
    
    # Part 2: fix a path and M, repeat experiment to get variance
    print("\nRepeating Monte Carlo experiments for fixed path and M:")
    path = random_paths[0]  # Use the first path
    M = 100
    repeats = 10
    
    S_10_estimates = []
    V_10_estimates = []
    
    for _ in range(repeats):
        # Estimate S_10
        S_10_estimate = model.monte_carlo_expectation(n, N, M, path)
        S_10_estimates.append(S_10_estimate)
        
        # Estimate V_10
        K = (1+r)**N * S0
        payoff_func = lambda S: model.european_call_payoff(S, K)
        V_10_estimate = model.monte_carlo_expectation(n, N, M, path, payoff_func)
        V_10_estimates.append(V_10_estimate)
    
    # Compute statistics
    S_10_mean = np.mean(S_10_estimates)
    S_10_variance = np.var(S_10_estimates)
    V_10_mean = np.mean(V_10_estimates)
    V_10_variance = np.var(V_10_estimates)
    
    exact_S_10 = model.compute_stock_price_from_path(path)
    
    print(f"Path: {path}")
    print(f"Exact S_10: {exact_S_10:.6f}")
    print(f"Mean S_10 estimate: {S_10_mean:.6f}")
    print(f"Variance of S_10 estimates: {S_10_variance:.6f}")
    print(f"Mean V_10 estimate: {V_10_mean:.6f}")
    print(f"Variance of V_10 estimates: {V_10_variance:.6f}")

# Part 2(d): Lookback option
def test_lookback_option():
    """Test Monte Carlo for lookback option"""
    print("\nPART 2(d): MONTE CARLO FOR LOOKBACK OPTION")
    print("----------------------------------------")

    S0 = 1.0
    r = 0.001
    u = 1.005
    d = 1.0001
    N = 100
    
    model = BAPMMonteCarlo(S0, r, u, d, N)
    print(f"Model parameters: S0={S0}, r={r}, u={u}, d={d}, N={N}")
    
    # Define M values to test
    M_values = [10, 100, 1000]
    
    print("\nEstimating lookback option price using Monte Carlo:")
    lookback_results = model.test_monte_carlo_convergence(M_values, option_type="lookback", repeat=5)
    

    print(f"{'M':>10} {'V0 Estimate':>15} {'Variance':>10}")
    for i, M in enumerate(M_values):
        estimate = lookback_results["means"][i]
        variance = lookback_results["variances"][i]
        print(f"{M:10d} {estimate:15.6f} {variance:10.6f}")
    
    # Generate a few random paths and compute lookback payoff
    print("\nLookback payoffs for different paths:")
    for i in range(5):
        # Generate a random path
        path = model.generate_random_path(N, model.p_tilde)
        
        # Compute lookback payoff
        payoff = model.lookback_option_payoff(path)
        
        print(f"Path {i+1}: {payoff:.6f}")

In [39]:
# %===============================%
# Run all tests
# %===============================%

# Part 2(a): Test with N=5
test_monte_carlo_N5()

# Part 2(b): Test with N=100
test_monte_carlo_N100()

# Part 2(c): Test with random paths
test_monte_carlo_random_paths()

# Part 2(d): Test lookback option
test_lookback_option()


PART 2(a): MONTE CARLO WITH N=5
-------------------------------
BAPM for Monte Carlo, Parameters: S0=1.0, r=0.05, u=1.1, d=1.01, N=5

Estimating S0 from Monte Carlo:
         M     S0 Estimate      Error   Variance
         1        1.319183   0.319183   0.019148
         5        1.280360   0.280360   0.001794
        10        1.292609   0.292609   0.001526
        32        1.271266   0.271266   0.000272
       100        1.277701   0.277701   0.000089
      1000        1.277740   0.277740   0.000008

Exact option price V0 = 0.039031

Estimating V0 from Monte Carlo:
         M     V0 Estimate      Error   Variance
         1        0.050877   0.011846   0.003667
         5        0.050258   0.011226   0.000593
        10        0.025167   0.013864   0.000041
        32        0.042617   0.003585   0.000144
       100        0.039764   0.000732   0.000020
      1000        0.039743   0.000712   0.000003

PART 2(b): MONTE CARLO WITH N=100, r=10^-3, u = 1+5x10^-3, d=1+10^-4
----------

-------------
## Question 2 - Responses
-------------
Part 2(a): 
 - The monte carlo estimates for S0 converge to ~1.27 as M increases. This is actually the expected terminal stock price, not S0. Also variance decreases like 1/M
 - Option price estimate is more precise for larger M when estimating V0 with Monte Carlo.

Part 2(b): Monte Carlo with N = 100
 - With N=100 Monte Carlo is essential since there are 2^100 possible paths. The method successfully estimates E(S_100) = 1.105 and V0 = 0.0075 with increasing precision for increasing M.

Part 2(c): Monte Carlo with random paths
 - Monte carlo produces precise estimates throughout the different scenarios. Although the values do not match since there is a consistent 0.1 different between the exact and estimated values.

Part 2(d): Lookback
 - Here we test Monte Carlo for a truly path dependent derivative. Using state variables M_n and S_n significantly reduces the computational complexity. We also get convergence for increasing M which suggests it models the path-dependent derivative properly.
